In [9]:
import pandas as pd
import plotly.graph_objects as go
import json

# ── Load files ────────────────────────────────────────────────
coords_df = pd.read_csv("hospital_coordinates.csv")       # Code, Name, Postcode, latitude, longitude
comm_df   = pd.read_csv("hospital_communities.csv")  # hospital (index), provider, propagation_risk_score etc
pdf = pd.read_csv("agg_hospital_transfers.csv")  # source, target, weight

# ── Check structures ──────────────────────────────────────────
print("Coordinates columns:", coords_df.columns.tolist())
print("Communities columns:", comm_df.columns.tolist())
print("Propagation columns:", pdf.columns.tolist())
print(f"\nCoords rows:    {len(coords_df)}")
print(f"comm rows:      {len(comm_df)}")

Coordinates columns: ['Code', 'Name', 'Postcode', 'latitude', 'longitude', 'geocoded']
Communities columns: ['hospital', 'provider', 'community_id', 'weighted_in_degree', 'weighted_out_degree', 'total_transfers', 'cross_community_bridges']
Propagation columns: ['source', 'target', 'out_provider', 'in_provider', 'weight']

Coords rows:    247
comm rows:      125


In [10]:
comm_df.head(10)

,hospital,provider,community_id,weighted_in_degree,weighted_out_degree,total_transfers,cross_community_bridges
0,R0A,MANCHESTER UNIVERSITY NHS FOUNDATION TRUST,0,27494,1569,29063,116
1,RWJ,STOCKPORT NHS FOUNDATION TRUST,0,2431,3226,5657,49
2,REM,AINTREE UNIVERSITY HOSPITAL NHS FOUNDATION TRUST,0,1840,839,2679,44
3,RXN,LANCASHIRE TEACHING HOSPITALS NHS FOUNDATION T...,0,1593,2177,3770,50
4,RBN,ST HELENS AND KNOWSLEY HOSPITAL SERVICES NHS T...,0,1229,737,1966,35
5,RXL,BLACKPOOL TEACHING HOSPITALS NHS FOUNDATION TRUST,0,1223,1813,3036,42
6,RBS,ALDER HEY CHILDREN'S NHS FOUNDATION TRUST,0,822,162,984,7
7,RMP,TAMESIDE AND GLOSSOP INTEGRATED CARE NHS FOUND...,0,614,6096,6710,13
8,RJR,COUNTESS OF CHESTER HOSPITAL NHS FOUNDATION TRUST,0,593,518,1111,20
9,RBT,MID CHESHIRE HOSPITALS NHS FOUNDATION TRUST,0,562,2376,2938,67


In [11]:
coords_df.head(5)

,Code,Name,Postcode,latitude,longitude,geocoded
0,RCF,AIREDALE NHS FOUNDATION TRUST,BD20 6TD,53.897957,-1.962729,True
1,RBS,ALDER HEY CHILDREN'S NHS FOUNDATION TRUST,L12 2AP,53.421263,-2.897796,True
2,RTK,ASHFORD AND ST PETER'S HOSPITALS NHS FOUNDATIO...,KT16 0PZ,51.377831,-0.527041,True
3,RVN,AVON AND WILTSHIRE MENTAL HEALTH PARTNERSHIP N...,BA1 3QE,51.388601,-2.392311,True
4,RF4,"BARKING, HAVERING AND REDBRIDGE UNIVERSITY HOS...",RM7 0AG,51.568649,0.179058,True


In [4]:
# ── Merge on ODS code ─────────────────────────────────────────
# coords_df uses "Code", vuln_df uses "hospital" (index reset to column)
merged = coords_df.merge(
    vuln_df,
    left_on="Code",
    right_on="hospital",
    how="inner"
)

print("Communities in data:", sorted(merged["community"].unique()))
print("Community counts:\n", merged["community"].value_counts().sort_index())
print("Missing names:", [c for c in merged["community"].unique() if c not in community_names])


Communities in data: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Community counts:
 community
0     10
1     18
2      3
3     28
4      7
5     20
6      9
7      8
8      7
9     12
10     3
Name: count, dtype: int64


NameError: name 'community_names' is not defined

In [ ]:
# ── Community colour map ──────────────────────────────────────
base_colors = [
    "#4e9af1", "#f97316", "#22c55e", "#e879f9", "#facc15",
    "#f43f5e", "#06b6d4", "#a3e635", "#c084fc", "#fb923c",
    "#34d399", "#f472b6", "#60a5fa", "#fbbf24", "#818cf8",
    "#2dd4bf", "#fb7185", "#a78bfa", "#4ade80", "#38bdf8"
]
community_ids       = sorted(merged["community"].unique())
community_color_map = {c: base_colors[i % len(base_colors)] for i, c in enumerate(community_ids)}

# ── Community names ───────────────────────────────────────────
community_names = {
    0:  "West Midlands",
    1:  "South West",
    2:  "Coventry and Warwickshire",
    3:  "West London and South East",
    4:  "Central and East London",
    5:  "North West",
    6:  "East of England",
    7:  "North East",
    8:  "East Midlands",
    9:  "West Yorkshire",
    10: "North Yorkshire and the Humber",
    11: "India"
}

# ── Risk tier ─────────────────────────────────────────────────
def risk_tier(score, sensitivity_pct):
    if sensitivity_pct >= 75:   return "Tier 1 — Critical"
    elif sensitivity_pct >= 50: return "Tier 2 — High Risk"
    elif sensitivity_pct >= 25: return "Tier 3 — Elevated"
    else:                       return "Tier 4 — Standard"

merged["risk_tier"] = merged.apply(
    lambda row: risk_tier(row["propagation_risk_score"], row["sensitivity_pct"]),
    axis=1
)

print("\nRisk tier distribution:")
print(merged["risk_tier"].value_counts().to_string())

# ── Build figure ──────────────────────────────────────────────
fig = go.Figure()

# ── Layer 1: ICB boundaries — single trace ────────────────────
all_lons = []
all_lats = []

for feature in icb_geojson["features"]:
    geo_type = feature["geometry"]["type"]
    coords   = feature["geometry"]["coordinates"]
    polygons = coords if geo_type == "MultiPolygon" else [coords]
    for polygon in polygons:
        for ring in polygon:
            all_lons += [pt[0] for pt in ring] + [None]
            all_lats += [pt[1] for pt in ring] + [None]

fig.add_trace(go.Scattermapbox(
    lon=all_lons,
    lat=all_lats,
    mode="lines",
    line=dict(color="#7a9bb5", width=0.8),
    hoverinfo="skip",
    showlegend=False,
    fill="none"
))

# ── Layer 2: Hospital nodes per community ─────────────────────
for comm_id in community_ids:
    comm  = merged[merged["community"] == comm_id]
    color = community_color_map[comm_id]

    fig.add_trace(go.Scattermapbox(
        lat=comm["latitude"],
        lon=comm["longitude"],
        mode="markers",
        name=f"{community_names.get(comm_id, f'Community {comm_id}')} (n={len(comm)})",
        marker=dict(
            size=4 + comm["propagation_risk_score"] * 12,
            color=color,
            opacity=0.85,
            sizemode="diameter"
        ),
        text=(
            "<b>" + comm["provider"]                                               + "</b><br>" +
            "Code: "          + comm["Code"]                                       + "<br>" +
            "Community: "     + comm["community"].map(community_names)             + "<br>" +
            "Risk Score: "    + comm["propagation_risk_score"].round(4).astype(str)+ "<br>" +
            "Risk Tier: "     + comm["risk_tier"]                                  + "<br>" +
            "Sensitivity %: " + comm["sensitivity_pct"].astype(str)                + "%<br>" +
            "Betweenness: "   + comm["betweenness"].round(4).astype(str)           + "<br>" +
            "In-Degree: "     + comm["in_degree"].round(4).astype(str)             + "<br>" +
            "Bridge Edges: "  + comm["cross_community_bridges"].astype(str)
        ),
        hoverinfo="text",
        hoverlabel=dict(bgcolor="white", bordercolor=color, font=dict(size=11))
    ))

# ── Layer 3: Tier 1 critical node labels ─────────────────────
tier1 = merged[merged["risk_tier"] == "Tier 1 — Critical"]
fig.add_trace(go.Scattermapbox(
    lat=tier1["latitude"],
    lon=tier1["longitude"],
    mode="markers+text",
    name="Tier 1 — Critical",
    marker=dict(
        size=20,
        color="rgba(200, 0, 0, 0.3)",
        opacity=1.0,
        sizemode="diameter"
    ),
    text=tier1["Code"],
    textposition="top right",
    textfont=dict(size=9, color="#cc0000", family="Arial Black"),
    hoverinfo="skip"
))

# ── Layout ────────────────────────────────────────────────────
fig.update_layout(
    mapbox=dict(
        style="carto-positron",
        center=dict(lat=52.5, lon=-1.8),
        zoom=5.8,
        layers=[dict(
            sourcetype="raster",
            source=["https://basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}.png"],
            below="traces"
        )]
    ),
    title=dict(
        text=(
            "Hospital Cyberattack Propagation Risk across England<br>"
            "<sup>Node size = Propagation Risk Score | "
            "Colour = Transfer Community | "
            "Red labels = Tier 1 Critical Nodes</sup>"
        ),
        font=dict(size=15),
        x=0.5
    ),
    legend=dict(
        title="Transfer Communities",
        bgcolor="white",
        bordercolor="#cccccc",
        borderwidth=1,
        font=dict(size=10)
    ),
    margin=dict(l=0, r=0, t=80, b=0),
    height=800
)

# ── Save and open ─────────────────────────────────────────────
fig.write_html("hospital_vulnerability_map.html")
print("Saved to hospital_vulnerability_map.html")

Total communities: 11 → [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


AssertionError: Expected 12, got 11

In [ ]:
from shapely.geometry import Point, shape

# ── Build ICB polygon lookup from GeoJSON ─────────────────────
icb_polygons = []
for feature in icb_geojson["features"]:
    props    = feature["properties"]
    geometry = shape(feature["geometry"])  # converts GeoJSON to shapely polygon
    icb_polygons.append({
        "icb23cd":  props["ICB23CD"],
        "icb23nm":  props["ICB23NM"],
        "geometry": geometry
    })

# ── Assign each hospital to an ICB ───────────────────────────
def find_icb(lat, lon):
    point = Point(lon, lat)  # shapely uses (lon, lat) order
    for icb in icb_polygons:
        if icb["geometry"].contains(point):
            return icb["icb23cd"], icb["icb23nm"]
    return None, None  # hospital outside all ICBs

merged["icb23cd"], merged["icb23nm"] = zip(*merged.apply(
    lambda row: find_icb(row["latitude"], row["longitude"]), axis=1
))

print(f"Matched to ICB: {merged['icb23cd'].notna().sum()} / {len(merged)}")
print(f"Unmatched:      {merged['icb23cd'].isna().sum()}")
print(merged[merged['icb23cd'].isna()][["Code", "provider", "latitude", "longitude"]])

Matched to ICB: 125 / 125
Unmatched:      0
Empty DataFrame
Columns: [Code, provider, latitude, longitude]
Index: []


In [ ]:
# ── Assign each ICB the community of its highest risk hospital ─
icb_community = (
    merged.sort_values("propagation_risk_score", ascending=False)
    .groupby("icb23cd")
    .agg(
        icb_name       =("icb23nm",               "first"),
        community_id   =("community",              "first"),
        top_hospital   =("provider",               "first"),
        top_code       =("Code",                   "first"),
        hospital_count =("Code",                   "count"),
        avg_risk       =("propagation_risk_score", "mean"),
        max_risk       =("propagation_risk_score", "max"),
    )
    .reset_index()
)

print(f"ICBs with hospitals: {len(icb_community)} / 42")
print(icb_community[["icb_name", "community_id", "top_hospital", "hospital_count"]].to_string(index=False))

# ── Community colour map ──────────────────────────────────────
base_colors = [
    "#4e9af1", "#f97316", "#22c55e", "#e879f9", "#facc15",
    "#f43f5e", "#06b6d4", "#a3e635", "#c084fc", "#fb923c",
    "#34d399", "#f472b6", "#60a5fa", "#fbbf24", "#818cf8",
    "#2dd4bf", "#fb7185", "#a78bfa", "#4ade80", "#38bdf8"
]
community_ids       = sorted(merged["community"].unique())
community_color_map = {c: base_colors[i % len(base_colors)] for i, c in enumerate(community_ids)}

def hex_to_rgba(hex_color, alpha=0.35):
    """Convert hex colour to rgba string with given opacity"""
    hex_color = hex_color.lstrip("#")
    r, g, b   = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"

icb_community["color"]      = icb_community["community_id"].map(community_color_map)
icb_community["fill_color"] = icb_community["color"].apply(hex_to_rgba)

# ── Build figure ──────────────────────────────────────────────
fig = go.Figure()

# ── Layer 1: ICB choropleth — fast single trace per community ─
from collections import defaultdict

# Group ICB polygon coordinates by community
community_lons = defaultdict(list)
community_lats = defaultdict(list)
community_hover = defaultdict(list)

for feature in icb_geojson["features"]:
    props    = feature["properties"]
    icb_code = props["ICB23CD"]
    geo_type = feature["geometry"]["type"]
    coords   = feature["geometry"]["coordinates"]
    polygons = coords if geo_type == "MultiPolygon" else [coords]

    # Look up community for this ICB
    icb_row = icb_community[icb_community["icb23cd"] == icb_code]
    
    if len(icb_row) == 0:
        comm_key   = "none"
        hover_text = f"<b>{props['ICB23NM']}</b><br>No hospitals in dataset"
    else:
        row        = icb_row.iloc[0]
        comm_key   = int(row["community_id"])
        hover_text = (
            f"<b>{props['ICB23NM']}</b><br>"
            f"Community: {community_names.get(int(row['community_id']), str(int(row['community_id'])))}<br>"
            f"Top hospital: {row['top_hospital']}<br>"
            f"Hospitals: {int(row['hospital_count'])}<br>"
            f"Avg risk: {row['avg_risk']:.4f}"
        )

    for polygon in polygons:
        for ring in polygon:
            community_lons[comm_key] += [pt[0] for pt in ring] + [None]
            community_lats[comm_key] += [pt[1] for pt in ring] + [None]

# Draw one trace per community
for comm_key in community_lons:
    if comm_key == "none":
        fill_color = "rgba(200,200,200,0.2)"
        line_color = "#aaaaaa"
        name       = "No data"
    else:
        hex_col    = community_color_map[comm_key]
        fill_color = hex_to_rgba(hex_col, alpha=0.35)
        line_color = "#555555"
        name       = f"Community {comm_key}"

    fig.add_trace(go.Scattermapbox(
        lon=community_lons[comm_key],
        lat=community_lats[comm_key],
        mode="lines",
        fill="toself",
        fillcolor=fill_color,
        line=dict(color=line_color, width=0.8),
        hoverinfo="skip",      # skip hover on boundaries — hospital nodes handle it
        showlegend=False,
        name=name
    ))

# ── Layer 2: Hospital nodes per community ─────────────────────
for comm_id in community_ids:
    comm  = merged[merged["community"] == comm_id]
    color = community_color_map[comm_id]

    fig.add_trace(go.Scattermapbox(
        lat=comm["latitude"],
        lon=comm["longitude"],
        mode="markers",
        name=f"Community {comm_id} (n={len(comm)})",
        marker=dict(
            size=4 + comm["propagation_risk_score"] * 12,
            color=color,
            opacity=0.9,
            sizemode="diameter"
        ),
        text=(
            "<b>" + comm["provider"] + "</b><br>" +
            "Code: "         + comm["Code"]                                        + "<br>" +
            "Community: "    + comm["community"].astype(str)                       + "<br>" +
            "Risk Score: "   + comm["propagation_risk_score"].round(4).astype(str) + "<br>" +
            "Betweenness: "  + comm["betweenness"].round(4).astype(str)            + "<br>" +
            "In-Degree: "    + comm["in_degree"].round(4).astype(str)              + "<br>" +
            "Bridge Edges: " + comm["cross_community_bridges"].astype(str)
        ),
        hoverinfo="text",
        hoverlabel=dict(bgcolor="white", bordercolor=color, font=dict(size=11))
    ))

# ── Layer 3: Tier 1 critical node labels ─────────────────────
tier1 = merged[merged["propagation_risk_score"] >= merged["propagation_risk_score"].quantile(0.75)]
fig.add_trace(go.Scattermapbox(
    lat=tier1["latitude"],
    lon=tier1["longitude"],
    mode="markers+text",
    name="Tier 1 — Critical",
    marker=dict(size=18, color="rgba(244, 63, 94, 0.25)", sizemode="diameter"),   # semi-transparent red fill
    text=tier1["Code"],
    textposition="top right",
    textfont=dict(size=9, color="#cc0000", family="Arial Black"),
    hoverinfo="skip"
))

# ── Layout ────────────────────────────────────────────────────
fig.update_layout(
    mapbox=dict(
        style="carto-positron",
        center=dict(lat=52.5, lon=-1.8),
        zoom=5.8,
        layers=[dict(
            sourcetype="raster",
            source=["https://basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}.png"],
            below="traces"
        )]
    ),
    title=dict(
        text=(
            "Hospital Transfer Network — ICB Regions Coloured by Louvain Community<br>"
            "<sup>ICB shading = dominant community | "
            "Node size = Propagation Risk Score | "
            "Red labels = Top 25% Risk Nodes</sup>"
        ),
        font=dict(size=14),
        x=0.5
    ),
    legend=dict(
        title="Louvain Communities",
        bgcolor="white",
        bordercolor="#cccccc",
        borderwidth=1,
        font=dict(size=10)
    ),
    margin=dict(l=0, r=0, t=80, b=0),
    height=800
)

# ── Save and open ─────────────────────────────────────────────
fig.write_html("hospital_icb_community_map.html")
webbrowser.open("file://" + os.path.abspath("hospital_icb_community_map.html"))
print("Saved to hospital_icb_community_map.html")

ICBs with hospitals: 42 / 42
                                                                     icb_name  community_id                                                  top_hospital  hospital_count
                            NHS Cheshire and Merseyside Integrated Care Board             5                   MID CHESHIRE HOSPITALS NHS FOUNDATION TRUST               9
                   NHS Staffordshire and Stoke-on-Trent Integrated Care Board             0          UNIVERSITY HOSPITAL OF NORTH STAFFORDSHIRE NHS TRUST               1
                     NHS Shropshire, Telford and Wrekin Integrated Care Board             0                 THE SHREWSBURY AND TELFORD HOSPITAL NHS TRUST               1
                                       NHS Lincolnshire Integrated Care Board             8                       UNITED LINCOLNSHIRE HOSPITALS NHS TRUST               1
              NHS Leicester, Leicestershire and Rutland Integrated Care Board             8                   UNIVERSITY 

/var/folders/rd/x1432m7n7db4ntw4kp5cf65h0000gn/T/ipykernel_78850/1606987403.py:91: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
/var/folders/rd/x1432m7n7db4ntw4kp5cf65h0000gn/T/ipykernel_78850/1606987403.py:108: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
/var/folders/rd/x1432m7n7db4ntw4kp5cf65h0000gn/T/ipykernel_78850/1606987403.py:108: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
/var/folders/rd/x1432m7n7db4ntw4kp5cf65h0000gn/T/ipykernel_78850/1606987403.py:108: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_t

Saved to hospital_icb_community_map.html


Python(89889) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [ ]:
import pandas as pd

# ── Load transfer data ────────────────────────────────────────
transfers = pd.read_csv("agg_hospital_transfers.csv")

# ── Build ICB community weight table ─────────────────────────
# For each ICB, sum actual transfer weights from hospitals in each community
rows = []

for feature in icb_geojson["features"]:
    props    = feature["properties"]
    icb_code = props["ICB23CD"]
    icb_name = props["ICB23NM"]

    # Hospitals in this ICB
    icb_hospitals = merged[merged["icb23cd"] == icb_code]["Code"].tolist()

    if len(icb_hospitals) == 0:
        rows.append({
            "icb_code":           icb_code,
            "icb_name":           icb_name,
            "total_hospitals":    0,
            "dominant_community": None,
            "dominance_pct":      None,
            "total_weight":       None,
            "num_communities":    None,
            "communities_present":None,
            "top_hospital":       None,
            "avg_risk":           None,
            "max_risk":           None,
        })
        continue

    # Get all transfers where source OR target is in this ICB
    icb_transfers = transfers[
        transfers["source"].isin(icb_hospitals) |
        transfers["target"].isin(icb_hospitals)
    ].copy()

    # Tag each transfer row with the community of the ICB hospital involved
    def get_community(row):
        # Prefer source hospital if in ICB, else target
        if row["source"] in icb_hospitals:
            return merged[merged["Code"] == row["source"]]["community"].values[0] \
                   if len(merged[merged["Code"] == row["source"]]) > 0 else None
        else:
            return merged[merged["Code"] == row["target"]]["community"].values[0] \
                   if len(merged[merged["Code"] == row["target"]]) > 0 else None

    icb_transfers["community"] = icb_transfers.apply(get_community, axis=1)
    icb_transfers = icb_transfers.dropna(subset=["community"])

    total_weight = icb_transfers["weight"].sum()

    # Per community breakdown
    comm_breakdown = (
        icb_transfers.groupby("community")
        .agg(
            transfer_weight =("weight", "sum"),
            transfer_count  =("weight", "count"),
        )
        .reset_index()
        .sort_values("transfer_weight", ascending=False)
    )
    comm_breakdown["weight_pct"] = (
        comm_breakdown["transfer_weight"] / total_weight * 100
    ).round(1)

    dominant = comm_breakdown.iloc[0]
    dom_pct  = dominant["weight_pct"]

    # Risk metrics for hospitals in this ICB
    icb_risk = merged[merged["icb23cd"] == icb_code]

    rows.append({
        "icb_code":            icb_code,
        "icb_name":            icb_name,
        "total_hospitals":     len(icb_hospitals),
        "dominant_community":  int(dominant["community"]),
        "dominance_pct":       round(dom_pct, 1),
        "dominant_weight":     int(dominant["transfer_weight"]),
        "total_weight":        int(total_weight),
        "num_communities":     len(comm_breakdown),
        "communities_present": ", ".join(comm_breakdown["community"].astype(str).tolist()),
        "avg_risk":            round(icb_risk["propagation_risk_score"].mean(), 4),
        "max_risk":            round(icb_risk["propagation_risk_score"].max(), 4),
        "top_hospital":        icb_risk.sort_values(
                                   "propagation_risk_score", ascending=False
                               ).iloc[0]["provider"]
    })

icb_table = pd.DataFrame(rows).sort_values("dominance_pct", ascending=False)

# ── Print summary table ───────────────────────────────────────
print("=== ICB Community Dominance Table (weighted by actual transfers) ===\n")
print(icb_table[icb_table["total_hospitals"] > 0][[
    "icb_name", "dominant_community", "dominance_pct",
    "total_weight", "total_hospitals", "num_communities",
    "avg_risk", "max_risk", "top_hospital", "communities_present"
]].to_string(index=False))

# ── Print full per-ICB breakdown ──────────────────────────────
print("\n\n=== Full Community Weight Breakdown per ICB ===\n")
for _, icb_row in icb_table[icb_table["total_hospitals"] > 0].iterrows():
    icb_hospitals = merged[merged["icb23cd"] == icb_row["icb_code"]]["Code"].tolist()

    icb_transfers = transfers[
        transfers["source"].isin(icb_hospitals) |
        transfers["target"].isin(icb_hospitals)
    ].copy()

    def get_community(row):
        if row["source"] in icb_hospitals:
            vals = merged[merged["Code"] == row["source"]]["community"].values
            return vals[0] if len(vals) > 0 else None
        else:
            vals = merged[merged["Code"] == row["target"]]["community"].values
            return vals[0] if len(vals) > 0 else None

    icb_transfers["community"] = icb_transfers.apply(get_community, axis=1)
    icb_transfers = icb_transfers.dropna(subset=["community"])
    total_weight  = icb_transfers["weight"].sum()

    comm_breakdown = (
        icb_transfers.groupby("community")
        .agg(
            transfer_weight =("weight", "sum"),
            transfer_count  =("weight", "count"),
        )
        .reset_index()
        .sort_values("transfer_weight", ascending=False)
    )
    comm_breakdown["weight_pct"] = (
        comm_breakdown["transfer_weight"] / total_weight * 100
    ).round(1)

    print(f"{'='*70}")
    print(f"ICB: {icb_row['icb_name']}")
    print(f"Hospitals: {int(icb_row['total_hospitals'])} | "
          f"Total transfers: {int(total_weight):,} | "
          f"Communities: {int(icb_row['num_communities'])}")
    print(f"{'='*70}")
    print(comm_breakdown[[
        "community", "transfer_weight", "weight_pct", "transfer_count"
    ]].to_string(index=False))
    print()

# ── Save ─────────────────────────────────────────────────────
icb_table.to_csv("icb_community_dominance.csv", index=False)
print("Saved to icb_community_dominance.csv")

=== ICB Community Dominance Table (weighted by actual transfers) ===

                                                                     icb_name  dominant_community  dominance_pct  total_weight  total_hospitals  num_communities  avg_risk  max_risk                                                  top_hospital communities_present
                            NHS Cheshire and Merseyside Integrated Care Board                   5          100.0         30702                9                1    0.1076    0.2743                   MID CHESHIRE HOSPITALS NHS FOUNDATION TRUST                   5
                                              NHS Devon Integrated Care Board                   1          100.0          9839                3                1    0.2509    0.4131                   ROYAL DEVON AND EXETER NHS FOUNDATION TRUST                   1
  NHS Bristol, North Somerset and South Gloucestershire Integrated Care Board                   1          100.0         27540               

In [ ]:
df = pd.read_csv("icb_community_dominance.csv")
print(df)

     icb_code                                           icb_name  \
0   E54000008  NHS Cheshire and Merseyside Integrated Care Board   
1   E54000037                    NHS Devon Integrated Care Board   
2   E54000039  NHS Bristol, North Somerset and South Gloucest...   
3   E54000040  NHS Bath and North East Somerset, Swindon and ...   
4   E54000041                   NHS Dorset Integrated Care Board   
5   E54000042  NHS Hampshire and Isle of Wight Integrated Car...   
6   E54000043          NHS Gloucestershire Integrated Care Board   
7   E54000044  NHS Buckinghamshire, Oxfordshire and Berkshire...   
8   E54000048  NHS Lancashire and South Cumbria Integrated Ca...   
9   E54000050  NHS North East and North Cumbria Integrated Ca...   
10  E54000054           NHS West Yorkshire Integrated Care Board   
11  E54000055  NHS Birmingham and Solihull Integrated Care Board   
12  E54000056  NHS Cambridgeshire and Peterborough Integrated...   
13  E54000057       NHS Greater Manchester Integ

In [ ]:
from shapely.geometry import shape
from shapely.ops import unary_union
import json

# ── Build community → merged polygon mapping ──────────────────
community_polygons = {}

for feature in icb_geojson["features"]:
    props    = feature["properties"]
    icb_code = props["ICB23CD"]
    geometry = shape(feature["geometry"])

    # Look up dominant community for this ICB
    icb_row = icb_table[icb_table["icb_code"] == icb_code]

    if len(icb_row) == 0 or icb_row.iloc[0]["total_hospitals"] == 0:
        comm_key = "none"
    else:
        comm_key = int(icb_row.iloc[0]["dominant_community"])

    if comm_key not in community_polygons:
        community_polygons[comm_key] = []
    community_polygons[comm_key].append(geometry)

# ── Dissolve (merge) ICB polygons by community ────────────────
dissolved = {}
for comm_key, polygons in community_polygons.items():
    try:
        dissolved[comm_key] = unary_union(polygons)
    except Exception as e:
        print(f"  Error dissolving community {comm_key}: {e}")

print(f"Dissolved {len(community_polygons)} community groups into merged regions")
for comm_key, poly in dissolved.items():
    print(f"  Community {comm_key}: {poly.geom_type}")

# ── Plot dissolved community regions ─────────────────────────
fig = go.Figure()

# ── Community name mapping ────────────────────────────────────
community_names = {
    0:  "West Midlands",
    1:  "South West",
    2:  "Coventry and Warwickshire",
    3:  "West London and South East",
    4:  "Central and East London",
    5:  "North West",
    6:  "East of England",
    7:  "North East",
    8:  "East Midlands",
    9:  "West Yorkshire",
    10: "North Yorkshire and the Humber",
    11: "India"
}

# ── Layer 1: Dissolved community regions ─────────────────────
for comm_key, geometry in dissolved.items():
    if comm_key == "none":
        fill_color = "rgba(200,200,200,0.2)"
        line_color = "#aaaaaa"
        comm_label = "No data"
    else:
        hex_col    = community_color_map[comm_key]
        fill_color = hex_to_rgba(hex_col, alpha=0.4)
        line_color = hex_to_rgba(hex_col, alpha=0.9)
        comm_label = community_names.get(comm_key, f"Community {comm_key}")

    # Handle both Polygon and MultiPolygon after dissolve
    if geometry.geom_type == "Polygon":
        geom_list = [geometry]
    else:
        geom_list = list(geometry.geoms)

    all_lons = []
    all_lats = []
    for geom in geom_list:
        # Exterior ring
        lons = list(geom.exterior.coords.xy[0])
        lats = list(geom.exterior.coords.xy[1])
        all_lons += lons + [None]
        all_lats += lats + [None]

        # Interior rings (holes)
        for interior in geom.interiors:
            lons = list(interior.coords.xy[0])
            lats = list(interior.coords.xy[1])
            all_lons += lons + [None]
            all_lats += lats + [None]

    # Get community stats for hover
    if comm_key != "none":
        comm_hospitals  = merged[merged["community"] == comm_key]
        comm_icbs       = icb_table[icb_table["dominant_community"] == comm_key]
        n_hospitals     = len(comm_hospitals)
        n_icbs          = len(comm_icbs)
        avg_risk        = comm_hospitals["propagation_risk_score"].mean()
        top_hospital    = comm_hospitals.sort_values(
                              "propagation_risk_score", ascending=False
                          ).iloc[0]["provider"]
        hover_text = (
            f"<b>{community_names.get(comm_key, f'Community {comm_key}')}</b><br>"
            f"Hospitals: {n_hospitals}<br>"
            f"ICBs: {n_icbs}<br>"
            f"Avg risk score: {avg_risk:.4f}<br>"
            f"Top hospital: {top_hospital}"
        )
    else:
        hover_text = "<b>No hospitals in dataset</b>"

    fig.add_trace(go.Scattermapbox(
        lon=all_lons,
        lat=all_lats,
        mode="lines",
        fill="toself",
        fillcolor=fill_color,
        line=dict(color=line_color, width=1.5),
        text=hover_text,
        hoverinfo="text",
        name=comm_label,
        showlegend=comm_key != "none"
    ))





# ── Layer 2: Faint ICB boundaries for reference ───────────────
all_lons = []
all_lats = []
for feature in icb_geojson["features"]:
    geo_type = feature["geometry"]["type"]
    coords   = feature["geometry"]["coordinates"]
    polygons = coords if geo_type == "MultiPolygon" else [coords]
    for polygon in polygons:
        for ring in polygon:
            all_lons += [pt[0] for pt in ring] + [None]
            all_lats += [pt[1] for pt in ring] + [None]

fig.add_trace(go.Scattermapbox(
    lon=all_lons,
    lat=all_lats,
    mode="lines",
    line=dict(color="rgba(100,100,100,0.3)", width=0.5),
    hoverinfo="skip",
    showlegend=False,
    name="ICB boundaries"
))

# ── Layout ────────────────────────────────────────────────────
fig.update_layout(
    mapbox=dict(
        style="carto-positron",
        center=dict(lat=52.5, lon=-1.8),
        zoom=5.8,
        layers=[dict(
            sourcetype="raster",
            source=["https://basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}.png"],
            below="traces"
        )]
    ),
    title=dict(
        text=(
            "Hospital Transfer Network — Merged Community Regions<br>"
            "<sup>Regions = dissolved ICBs by dominant transfer community | "
        ),
        font=dict(size=14),
        x=0.5
    ),
    legend=dict(
        title="Transfer Communities",
        bgcolor="white",
        bordercolor="#cccccc",
        borderwidth=1,
        font=dict(size=10)
    ),
    margin=dict(l=0, r=0, t=80, b=0),
    height=800
)

# ── Save and open ─────────────────────────────────────────────
fig.write_html("hospital_community_regions_map.html")
webbrowser.open("file://" + os.path.abspath("hospital_community_regions_map.html"))
print("Saved to hospital_community_regions_map.html")

Dissolved 11 community groups into merged regions
  Community 5: MultiPolygon
  Community 0: Polygon
  Community 8: MultiPolygon
  Community 2: Polygon
  Community 6: MultiPolygon
  Community 3: MultiPolygon
  Community 4: Polygon
  Community 1: MultiPolygon
  Community 7: MultiPolygon
  Community 10: MultiPolygon
  Community 9: Polygon


/var/folders/rd/x1432m7n7db4ntw4kp5cf65h0000gn/T/ipykernel_78850/3476542638.py:110: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
/var/folders/rd/x1432m7n7db4ntw4kp5cf65h0000gn/T/ipykernel_78850/3476542638.py:139: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
/var/folders/rd/x1432m7n7db4ntw4kp5cf65h0000gn/T/ipykernel_78850/3476542638.py:139: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(


Saved to hospital_community_regions_map.html


Python(89891) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [ ]:
# ── Community summary table ───────────────────────────────────
community_summary = []

for comm_id in sorted(community_ids):
    comm_hospitals = merged[merged["community"] == comm_id].copy()
    
    # ICBs dominated by this community
    dominated_icbs = icb_table[
        (icb_table["dominant_community"] == comm_id) &
        (icb_table["total_hospitals"] > 0)
    ]

    # Total transfers within community (source AND target in same community)
    comm_codes = comm_hospitals["Code"].tolist()
    internal_transfers = transfers[
        transfers["source"].isin(comm_codes) &
        transfers["target"].isin(comm_codes)
    ]["weight"].sum()

    # Total transfers crossing INTO community from outside
    incoming_transfers = transfers[
        (~transfers["source"].isin(comm_codes)) &
        transfers["target"].isin(comm_codes)
    ]["weight"].sum()

    # Total transfers leaving community to outside
    outgoing_transfers = transfers[
        transfers["source"].isin(comm_codes) &
        (~transfers["target"].isin(comm_codes))
    ]["weight"].sum()

    # Top hospital by risk score
    top_hospital = comm_hospitals.sort_values(
        "propagation_risk_score", ascending=False
    ).iloc[0]

    community_summary.append({
        "community":            comm_id,
        "hospitals":            len(comm_hospitals),
        "icbs_dominated":       len(dominated_icbs),
        "icb_names":            " | ".join(dominated_icbs["icb_name"].tolist()),
        "internal_transfers":   int(internal_transfers),
        "incoming_transfers":   int(incoming_transfers),
        "outgoing_transfers":   int(outgoing_transfers),
        "total_transfers":      int(internal_transfers + incoming_transfers + outgoing_transfers),
        "avg_risk":             round(comm_hospitals["propagation_risk_score"].mean(), 4),
        "max_risk":             round(comm_hospitals["propagation_risk_score"].max(), 4),
        "avg_betweenness":      round(comm_hospitals["betweenness"].mean(), 4),
        "avg_in_degree":        round(comm_hospitals["in_degree"].mean(), 4),
        "bridge_edges":         int(comm_hospitals["cross_community_bridges"].sum()),
        "top_hospital_code":    top_hospital["Code"],
        "top_hospital_name":    top_hospital["provider"],
        "top_hospital_risk":    round(top_hospital["propagation_risk_score"], 4),
    })

comm_table = pd.DataFrame(community_summary)

# ── Print full table ──────────────────────────────────────────
print("=== Community Summary Table ===\n")
print(comm_table[[
    "community", "hospitals", "icbs_dominated",
    "internal_transfers", "incoming_transfers", "outgoing_transfers",
    "avg_risk", "max_risk", "avg_betweenness",
    "bridge_edges", "top_hospital_code", "top_hospital_name"
]].to_string(index=False))

# ── Print per community detail ────────────────────────────────
print("\n\n=== Per Community Detail ===\n")
for _, row in comm_table.iterrows():
    comm_hospitals = merged[merged["community"] == row["community"]] \
                     .sort_values("propagation_risk_score", ascending=False)

    print(f"{'='*75}")
    print(f"COMMUNITY {row['community']}  |  "
          f"{row['hospitals']} hospitals  |  "
          f"{row['icbs_dominated']} ICBs dominated  |  "
          f"Avg risk: {row['avg_risk']}  |  "
          f"Bridge edges: {row['bridge_edges']}")
    print(f"ICBs: {row['icb_names']}")
    print(f"{'='*75}")
    print(f"{'Code':<8} {'Risk Score':>12} {'Sensitivity':>12} "
          f"{'Betweenness':>12} {'In-Degree':>10} {'Provider'}")
    print(f"{'-'*75}")
    for _, hosp in comm_hospitals.iterrows():
        print(f"{hosp['Code']:<8} "
              f"{hosp['propagation_risk_score']:>12.4f} "
              f"{hosp['sensitivity_pct']:>11.1f}% "
              f"{hosp['betweenness']:>12.4f} "
              f"{hosp['in_degree']:>10.4f} "
              f"{hosp['provider'][:40]}")
    print(f"\nTransfers — Internal: {row['internal_transfers']:,} | "
          f"Incoming: {row['incoming_transfers']:,} | "
          f"Outgoing: {row['outgoing_transfers']:,}\n")

# ── Save ──────────────────────────────────────────────────────
comm_table.to_csv("community_summary_table.csv", index=False)
print("Saved to community_summary_table.csv")

=== Community Summary Table ===

 community  hospitals  icbs_dominated  internal_transfers  incoming_transfers  outgoing_transfers  avg_risk  max_risk  avg_betweenness  bridge_edges top_hospital_code                                        top_hospital_name
         0         10               5               70664               14913               10608    0.2545    0.7854           0.0071           715               RRK     UNIVERSITY HOSPITALS BIRMINGHAM NHS FOUNDATION TRUST
         1         18               8               50099               11184               11886    0.2033    0.4472           0.0102          1010               REF                       ROYAL CORNWALL HOSPITALS NHS TRUST
         2          3               1               53098                8094                5508    0.3215    0.5712           0.0078           291               RKB UNIVERSITY HOSPITALS COVENTRY AND WARWICKSHIRE NHS TRUST
         3         28              10              372329              